# IsoCLIP on Kaggle with Caltech101 and CUB-200-2011

This notebook reproduces the main IsoCLIP image-to-image retrieval pipeline on both Caltech101 and CUB-200-2011:

1. Prepare the Kaggle environment and IsoCLIP source code.
2. Download Caltech101 and CUB-200-2011 from Caltech Data and place them under the correct `--dataroot`.
3. Validate the project dataset loaders for `caltech101` and `cub2011`.
4. Run image-to-image retrieval with the CLIP baseline for both datasets.
5. Run image-to-image retrieval with IsoCLIP for both datasets.
6. Read `summary.csv` files and compare the metrics by dataset and method.

GPU and Internet access are recommended in Kaggle Notebook Settings.


## 0. General Configuration

The project loaders expect the datasets in this layout:

```text
/kaggle/working/datasets/
+-- caltech-101/
|   +-- 101_ObjectCategories/
+-- CUB_200_2011/
    +-- images.txt
    +-- image_class_labels.txt
    +-- train_test_split.txt
    +-- images/
```

`DATA_ROOT` points to `/kaggle/working/datasets`. The pipeline runs the same CLIP and IsoCLIP settings for `caltech101` and `cub2011`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import tarfile
import zipfile

IS_KAGGLE = Path('/kaggle/working').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
DATA_ROOT = WORK_DIR / 'datasets'
CALTECH_DIR = DATA_ROOT / 'caltech-101'
CALTECH_IMAGE_DIR = CALTECH_DIR / '101_ObjectCategories'
CUB_DIR = DATA_ROOT / 'CUB_200_2011'

DATASET_CONFIGS = [
    {
        'dataset_name': 'caltech101',
        'display_name': 'Caltech101',
        'dataset_dir': CALTECH_DIR,
        'query_split': 'test',
        'gallery_split': 'train',
        'baseline_out': 'kaggle_caltech101_clip_baseline',
        'isoclip_out': 'kaggle_caltech101_isoclip',
    },
    {
        'dataset_name': 'cub2011',
        'display_name': 'CUB-200-2011',
        'dataset_dir': CUB_DIR,
        'query_split': 'all',
        'gallery_split': 'all',
        'baseline_out': 'kaggle_cub2011_clip_baseline',
        'isoclip_out': 'kaggle_cub2011_isoclip',
    },
]
DATASET_NAMES = [cfg['dataset_name'] for cfg in DATASET_CONFIGS]

CLIP_MODEL_NAME = 'ViT-B/32'
ISO_KTOP = 150
ISO_KBOTTOM = 50
RUN_BASELINE = True
RUN_ISOCLIP = True
FORCE_CPU_RETRIEVAL = False
AUTO_CPU_FALLBACK_FOR_INCOMPATIBLE_GPU = True

DATA_ROOT.mkdir(parents=True, exist_ok=True)

print('WORK_DIR:', WORK_DIR)
print('DATA_ROOT:', DATA_ROOT)
print('CALTECH_DIR:', CALTECH_DIR)
print('CUB_DIR:', CUB_DIR)
print('DATASET_NAMES:', DATASET_NAMES)
print('CLIP_MODEL_NAME:', CLIP_MODEL_NAME)
print('FORCE_CPU_RETRIEVAL:', FORCE_CPU_RETRIEVAL)


## 1. Prepare the IsoCLIP Source Code

If this notebook is already inside this repository, it uses the current repository. If the notebook is run standalone on Kaggle, this cell clones `https://github.com/toanthangO20/IsoCLIP_ImageRetrieval` into `/kaggle/working/IsoCLIP_ImageRetrieval`.

In [ ]:
def run(cmd, cwd=None, check=True, env=None):
    cmd = [str(x) for x in cmd]
    print('> ' + ' '.join(cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check, env=env)

candidate_dirs = [Path.cwd(), WORK_DIR / 'IsoCLIP_ImageRetrieval']
PROJECT_DIR = None

for candidate in candidate_dirs:
    if (candidate / 'src' / 'retrieval.py').exists():
        PROJECT_DIR = candidate.resolve()
        break

if PROJECT_DIR is None:
    PROJECT_DIR = WORK_DIR / 'IsoCLIP_ImageRetrieval'
    if not PROJECT_DIR.exists():
        run(['git', 'clone', 'https://github.com/toanthangO20/IsoCLIP_ImageRetrieval.git', str(PROJECT_DIR)])

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

print('PROJECT_DIR:', PROJECT_DIR)
print('Current directory:', Path.cwd())

## 2. Install Dependencies

This cell installs the packages required for the Caltech101 and CUB retrieval pipeline: Dassl, OpenAI CLIP, OpenCLIP, TorchMetrics, and auxiliary dependencies. Kaggle already ships with PyTorch and core packages such as `numpy`, `pandas`, `scikit-learn`, and `tqdm`, so this notebook does not downgrade them in order to avoid conflicts with the Kaggle environment.


In [ ]:
def import_ok(module_name):
    try:
        module = __import__(module_name)
        print(f'{module_name}:', getattr(module, '__version__', 'import ok'))
        return True
    except Exception as exc:
        print(f'{module_name} import failed: {type(exc).__name__}: {exc}')
        return False

if not (import_ok('numpy') and import_ok('scipy')):
    print('Repairing NumPy/SciPy binary stack for the current Kaggle image...')
    run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--force-reinstall', '--no-cache-dir',
        'numpy>=2.0,<2.3',
        'scipy>=1.14,<1.16',
    ])
    raise SystemExit('NumPy/SciPy were reinstalled. Restart the Kaggle session/kernel, then run the notebook again from the first cell.')

# Kaggle ships with recent numpy/pandas/sklearn/tqdm. Do not downgrade them:
# downgrading these core packages triggers many resolver conflict warnings and can break
# unrelated Kaggle packages. The CUB retrieval pipeline only needs the packages below.
base_packages = [
    'wheel',
    'PyYAML>=6.0.3,<6.1',
    'dotmap==1.3.30',
    'torchmetrics==1.4.0.post0',
    'openpyxl==3.1.2',
    'tabulate==0.9.0',
    'gdown==5.2.0',
    'easydict==1.13',
    'yacs==0.1.8',
    'ftfy',
    'regex',
    'timm',
    'open-clip-torch==3.2.0',
]

run([sys.executable, '-m', 'pip', 'install', '-q'] + base_packages)
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/KaiyangZhou/Dassl.pytorch'])

clip_specs = [
    'git+https://github.com/openai/CLIP.git@jongwook/issue-396',
    'git+https://github.com/openai/CLIP.git',
]

for spec in clip_specs:
    try:
        run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', spec])
        print('Installed CLIP from:', spec)
        break
    except subprocess.CalledProcessError:
        print('CLIP install failed for:', spec)
else:
    raise RuntimeError('Could not install OpenAI CLIP')

## 3. Download and Extract Caltech101 and CUB-200-2011

The primary source is Caltech Data. If you attached Kaggle datasets containing `caltech-101.zip` or `CUB_200_2011.tgz`, the notebook uses those files to avoid downloading them again. Otherwise, it downloads the archives into `/kaggle/working/datasets`.


In [ ]:
ARCHIVE_CONFIGS = {
    'caltech101': {
        'display_name': 'Caltech101',
        'path': DATA_ROOT / 'caltech-101.zip',
        'attached_names': ['caltech-101.zip', 'caltech101.zip'],
        'urls': [
            'https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1',
            'https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip',
        ],
        'min_bytes': 50_000_000,
    },
    'cub2011': {
        'display_name': 'CUB-200-2011',
        'path': DATA_ROOT / 'CUB_200_2011.tgz',
        'attached_names': ['CUB_200_2011.tgz', 'cub_200_2011.tgz'],
        'urls': [
            'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1',
            'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz',
        ],
        'min_bytes': 1_000_000_000,
    },
}

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def archive_looks_valid(path, min_bytes):
    return Path(path).exists() and Path(path).stat().st_size > min_bytes


def find_kaggle_input_file(filenames):
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for filename in filenames:
        matches = sorted(input_root.rglob(filename))
        if matches:
            return matches[0]
    return None


def download_or_copy_archive(key):
    info = ARCHIVE_CONFIGS[key]
    archive_path = info['path']
    if archive_looks_valid(archive_path, info['min_bytes']):
        print(f"{info['display_name']} archive already exists:", archive_path)
        return archive_path

    attached_archive = find_kaggle_input_file(info['attached_names'])
    if attached_archive is not None:
        print(f"Copy {info['display_name']} archive from Kaggle input:", attached_archive)
        shutil.copy2(attached_archive, archive_path)
        if archive_looks_valid(archive_path, info['min_bytes']):
            return archive_path
        raise RuntimeError(f"Attached {info['display_name']} archive looks too small: {archive_path}")

    for url in info['urls']:
        tmp_path = archive_path.with_suffix(archive_path.suffix + '.tmp')
        if tmp_path.exists():
            tmp_path.unlink()
        try:
            run(['curl', '-L', '--fail', '--retry', '3', '--retry-delay', '5', '-o', str(tmp_path), url])
            if archive_looks_valid(tmp_path, info['min_bytes']):
                tmp_path.replace(archive_path)
                print(f"Downloaded {info['display_name']} archive:", archive_path)
                return archive_path
            print(f"Downloaded {info['display_name']} file is too small, trying another URL")
        except subprocess.CalledProcessError:
            print(f"Download failed for {info['display_name']}, trying another URL:", url)
        finally:
            if tmp_path.exists():
                tmp_path.unlink()

    raise RuntimeError(
        f"Could not download {info['display_name']}. Enable Kaggle Internet or attach the archive as a Kaggle dataset."
    )


def is_within_directory(directory, target):
    directory = Path(directory).resolve()
    target = Path(target).resolve()
    return target == directory or str(target).startswith(str(directory) + os.sep)


def safe_extract_zip(zip_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = dest_dir / member.filename
            if not is_within_directory(dest_dir, target):
                raise RuntimeError(f'Unsafe zip member path: {member.filename}')
        zf.extractall(dest_dir)


def safe_extract_tar(tar_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, mode='r:*') as tf:
        for member in tf.getmembers():
            target = dest_dir / member.name
            if not is_within_directory(dest_dir, target):
                raise RuntimeError(f'Unsafe tar member path: {member.name}')
        tf.extractall(dest_dir)


def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path)
    name = archive_path.name.lower()
    print(f'Extracting {archive_path.name} -> {dest_dir}')
    if name.endswith('.zip'):
        safe_extract_zip(archive_path, dest_dir)
    elif name.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tbz2')):
        safe_extract_tar(archive_path, dest_dir)
    else:
        raise ValueError(f'Unsupported archive format: {archive_path}')


def find_dir(root, dirname):
    root = Path(root)
    for candidate in root.rglob(dirname):
        if candidate.is_dir():
            return candidate
    return None


def has_images(path):
    path = Path(path)
    return path.exists() and any(p.suffix.lower() in IMG_EXTS for p in path.rglob('*'))


def ensure_caltech101_layout():
    if CALTECH_IMAGE_DIR.exists() and has_images(CALTECH_IMAGE_DIR):
        return CALTECH_IMAGE_DIR

    existing = find_dir(DATA_ROOT, '101_ObjectCategories')
    if existing is not None and has_images(existing):
        CALTECH_DIR.mkdir(parents=True, exist_ok=True)
        if existing.resolve() != CALTECH_IMAGE_DIR.resolve():
            print('Moving Caltech101 image folder into project layout:', existing, '->', CALTECH_IMAGE_DIR)
            shutil.move(str(existing), str(CALTECH_IMAGE_DIR))
        return CALTECH_IMAGE_DIR

    raise RuntimeError('Could not locate Caltech101 101_ObjectCategories after extraction.')


def ensure_cub2011_layout():
    if (CUB_DIR / 'images.txt').exists() and (CUB_DIR / 'images').exists():
        return CUB_DIR

    existing = find_dir(DATA_ROOT, 'CUB_200_2011')
    if existing is not None and (existing / 'images.txt').exists() and (existing / 'images').exists():
        if existing.resolve() != CUB_DIR.resolve():
            print('Moving CUB folder into project layout:', existing, '->', CUB_DIR)
            shutil.move(str(existing), str(CUB_DIR))
        return CUB_DIR

    raise RuntimeError('Could not locate CUB_200_2011 metadata after extraction.')


def prepare_caltech101():
    if CALTECH_IMAGE_DIR.exists() and has_images(CALTECH_IMAGE_DIR):
        print('Caltech101 already extracted:', CALTECH_IMAGE_DIR)
        return CALTECH_IMAGE_DIR

    archive_path = download_or_copy_archive('caltech101')
    extract_archive(archive_path, DATA_ROOT)

    if not (CALTECH_IMAGE_DIR.exists() and has_images(CALTECH_IMAGE_DIR)):
        nested_names = {'101_objectcategories.tar.gz', '101_objectcategories.tgz', '101_objectcategories.tar'}
        nested_archives = [p for p in DATA_ROOT.rglob('*') if p.is_file() and p.name.lower() in nested_names]
        for nested_archive in nested_archives:
            extract_archive(nested_archive, CALTECH_DIR)
            if CALTECH_IMAGE_DIR.exists() and has_images(CALTECH_IMAGE_DIR):
                break

    image_root = ensure_caltech101_layout()
    print('Caltech101 image root:', image_root)
    return image_root


def prepare_cub2011():
    if (CUB_DIR / 'images.txt').exists() and (CUB_DIR / 'images').exists():
        print('CUB-200-2011 already extracted:', CUB_DIR)
        return CUB_DIR

    archive_path = download_or_copy_archive('cub2011')
    extract_archive(archive_path, DATA_ROOT)

    cub_root = ensure_cub2011_layout()
    print('CUB-200-2011 root:', cub_root)
    return cub_root


caltech_root = prepare_caltech101()
cub_root = prepare_cub2011()

print('Done.')
print('CALTECH_IMAGE_DIR:', CALTECH_IMAGE_DIR)
print('CUB_DIR:', CUB_DIR)


## 4. Validate the Project Datasets and Loaders

This cell checks the required metadata/files, verifies basic image counts, and calls the project loaders for both `caltech101` and `cub2011`. The Caltech101 loader creates `split_zhou_Caltech101.json` on the first run if it does not exist yet.


In [ ]:
import pandas as pd
from data_utils import get_dataset

required_cub_files = [
    CUB_DIR / 'images.txt',
    CUB_DIR / 'image_class_labels.txt',
    CUB_DIR / 'train_test_split.txt',
    CUB_DIR / 'images',
]

missing = [str(path) for path in required_cub_files if not path.exists()]
assert not missing, 'Missing CUB files: ' + ', '.join(missing)

num_cub_images = sum(1 for _ in (CUB_DIR / 'images').glob('*/*.jpg'))
print('CUB jpg images:', num_cub_images)
assert num_cub_images == 11788, f'Expected 11788 CUB images, found {num_cub_images}'

num_caltech_images = sum(1 for p in CALTECH_IMAGE_DIR.rglob('*') if p.suffix.lower() in IMG_EXTS)
print('Caltech101 image files:', num_caltech_images)
assert num_caltech_images > 0, 'Expected Caltech101 images, found none'

validation_datasets = {}
for cfg in DATASET_CONFIGS:
    dataset_name = cfg['dataset_name']
    splits = sorted({cfg['query_split'], cfg['gallery_split']})
    for split in splits:
        dataset = get_dataset(DATA_ROOT, dataset_name, split, preprocess=lambda image: image)
        validation_datasets[(dataset_name, split)] = dataset
        labels = dataset.get_labels()
        print(f"{cfg['display_name']} split={split}: {len(dataset)} samples, {len(set(labels))} classes")
        assert len(dataset) > 0, f'{dataset_name} {split} split is empty'

cub_all_dataset = validation_datasets[('cub2011', 'all')]
print('First CUB metadata row:')
display(cub_all_dataset.data.head(1))

caltech_preview_rows = []
for split in sorted({DATASET_CONFIGS[0]['query_split'], DATASET_CONFIGS[0]['gallery_split']}):
    dataset = validation_datasets[('caltech101', split)]
    item = dataset.data[0]
    caltech_preview_rows.append({
        'split': split,
        'impath': item.impath,
        'label': item.label,
        'classname': item.classname,
    })
print('First Caltech101 samples by split:')
display(pd.DataFrame(caltech_preview_rows))


In [ ]:
from PIL import Image


def sample_path_for_dataset(dataset_name, dataset, index=0):
    if dataset_name == 'cub2011':
        sample = dataset.data.iloc[index]
        return CUB_DIR / 'images' / sample.filepath
    if dataset_name == 'caltech101':
        return Path(dataset.data[index].impath)
    raise ValueError(f'Unsupported dataset for sample display: {dataset_name}')


for cfg in DATASET_CONFIGS:
    split = cfg['query_split']
    dataset = validation_datasets[(cfg['dataset_name'], split)]
    sample_path = sample_path_for_dataset(cfg['dataset_name'], dataset, index=0)
    print(f"{cfg['display_name']} sample ({split} split):", sample_path)
    display(Image.open(sample_path).convert('RGB').resize((256, 256)))


## 5. Run Baseline CLIP Image-to-Image Retrieval

The baseline uses standard CLIP features and disables IsoCLIP with `--no_iso`. Results are written to one folder per dataset:

- `results/kaggle_caltech101_clip_baseline/exp_*/summary.csv`
- `results/kaggle_cub2011_clip_baseline/exp_*/summary.csv`

The first run downloads the pretrained CLIP weights and extracts features. Features are cached under the project `data/image_features/...` directory.

Note that Kaggle sometimes assigns a Tesla P100 with `sm_60`, while some newer PyTorch images only build kernels for `sm_70+`. The cell below detects that case and runs retrieval on CPU to avoid the `no kernel image is available` error. To run faster on GPU, switch to a T4 or a PyTorch image compatible with P100 before rerunning.


In [ ]:
_RETRIEVAL_FORCE_CPU = None


def should_use_cpu_for_retrieval():
    if FORCE_CPU_RETRIEVAL:
        print('FORCE_CPU_RETRIEVAL=True, running retrieval on CPU')
        return True
    if not AUTO_CPU_FALLBACK_FOR_INCOMPATIBLE_GPU:
        return False
    try:
        import torch
        if not torch.cuda.is_available():
            print('CUDA is not available, retrieval.py will run on CPU')
            return False
        gpu_name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        arch = f'sm_{capability[0]}{capability[1]}'
        arch_list = torch.cuda.get_arch_list()
        print('Detected GPU:', gpu_name, 'capability:', capability, 'required arch:', arch)
        print('PyTorch CUDA arch list:', arch_list)
        if arch not in arch_list:
            print(f'GPU arch {arch} is not supported by this PyTorch build. Running retrieval.py on CPU.')
            return True
        return False
    except Exception as exc:
        print('Could not validate CUDA compatibility:', repr(exc))
        print('Running retrieval.py on CPU to avoid CUDA kernel-image errors.')
        return True


def retrieval_env():
    global _RETRIEVAL_FORCE_CPU
    if _RETRIEVAL_FORCE_CPU is None:
        _RETRIEVAL_FORCE_CPU = should_use_cpu_for_retrieval()
    env = os.environ.copy()
    if _RETRIEVAL_FORCE_CPU:
        env['CUDA_VISIBLE_DEVICES'] = ''
    return env


def run_retrieval(dataset_config, no_iso=False):
    out_path = dataset_config['baseline_out'] if no_iso else dataset_config['isoclip_out']
    method = 'CLIP baseline' if no_iso else 'IsoCLIP'
    print(f"\n=== {method}: {dataset_config['display_name']} ===")
    cmd = [
        sys.executable,
        str(PROJECT_DIR / 'src' / 'retrieval.py'),
        '--dataroot', str(DATA_ROOT),
        '--dataset_name', dataset_config['dataset_name'],
        '--clip_model_name', CLIP_MODEL_NAME,
        '--query_eval_type', 'image',
        '--gallery_eval_type', 'image',
        '--query_split', dataset_config['query_split'],
        '--gallery_split', dataset_config['gallery_split'],
        '--iso_ktop', str(ISO_KTOP),
        '--iso_kbottom', str(ISO_KBOTTOM),
        '--out_path', out_path,
    ]
    if no_iso:
        cmd.append('--no_iso')

    run(cmd, cwd=PROJECT_DIR, env=retrieval_env())


if RUN_BASELINE:
    for dataset_config in DATASET_CONFIGS:
        run_retrieval(dataset_config, no_iso=True)
else:
    print('RUN_BASELINE=False, skip baseline runs')


## 6. Run IsoCLIP Image-to-Image Retrieval

IsoCLIP uses the same datasets and model, but it uses pre-projection image features. The IsoCLIP projector is built from the SVD of the inter-modal operator `Psi = W_image.T @ W_text`, then removes the `ISO_KTOP` largest singular directions and the `ISO_KBOTTOM` smallest singular directions.

For `ViT-B/32`, the original project script uses `ISO_KTOP=150` and `ISO_KBOTTOM=50`.


In [ ]:
if RUN_ISOCLIP:
    for dataset_config in DATASET_CONFIGS:
        run_retrieval(dataset_config, no_iso=False)
else:
    print('RUN_ISOCLIP=False, skip IsoCLIP runs')


## 7. Inspect the IsoCLIP Projection in Code

This cell is not required for computing metrics, but it shows the core pipeline step: load the image/text projectors, build the IsoCLIP projectors, and inspect the retained subspace dimensionality.

In [ ]:
import torch

from utils import load_clip
from encode_no_projection import get_projection_layers
from retrieval import apply_iso

def safe_torch_device():
    if FORCE_CPU_RETRIEVAL:
        return torch.device('cpu')
    try:
        if torch.cuda.is_available():
            capability = torch.cuda.get_device_capability(0)
            arch = f'sm_{capability[0]}{capability[1]}'
            if arch in torch.cuda.get_arch_list():
                return torch.device('cuda')
            print(f'Using CPU in this cell because GPU arch {arch} is unsupported by the current PyTorch build.')
    except Exception as exc:
        print('CUDA compatibility check failed, using CPU:', repr(exc))
    return torch.device('cpu')

device = safe_torch_device()
print('Device for projector inspection:', device)
clip_model, normalized_model_name, _ = load_clip(CLIP_MODEL_NAME, None, False, device)
W_image, W_text = get_projection_layers(clip_model, normalized_model_name)

W_image_for_iso = W_image.T
W_text_for_iso = W_text.T
W_text_iso, W_image_iso = apply_iso(W_text_for_iso, W_image_for_iso, ISO_KTOP, ISO_KBOTTOM)

rank = min(W_image_for_iso.shape[1], W_text_for_iso.shape[1])
kept = rank - ISO_KTOP - ISO_KBOTTOM

print('Model:', normalized_model_name)
print('W_image:', tuple(W_image.shape))
print('W_text:', tuple(W_text.shape))
print('W_image_iso:', tuple(W_image_iso.shape))
print('W_text_iso:', tuple(W_text_iso.shape))
print('Retained singular directions:', kept, '/', rank)

## 8. Read Benchmark Results

The main retrieval metrics for each dataset are:

- `mAP`: mean Average Precision.
- `mAP_at_R`: AP at R, where R is the number of relevant items for the query.
- `precision_at_R`: R-precision.
- `recall_at_1`: whether the top-1 result has the same class as the query.


In [ ]:
import pandas as pd

result_roots = []
for cfg in DATASET_CONFIGS:
    result_roots.extend([cfg['baseline_out'], cfg['isoclip_out']])

summary_paths = []
for result_root in result_roots:
    summary_paths.extend(sorted((PROJECT_DIR / 'results' / result_root).glob('exp_*/summary.csv')))
summary_paths = sorted(summary_paths)

print('Found summary files:', len(summary_paths))
for path in summary_paths:
    print(path)

assert summary_paths, 'No summary.csv found. Run the retrieval cells first.'

summary_df = pd.concat([pd.read_csv(path).assign(source_file=str(path)) for path in summary_paths], ignore_index=True)
columns = [
    'dataset_name',
    'clip_model_name',
    'query_split',
    'gallery_split',
    'no_iso',
    'iso_ktop',
    'iso_kbottom',
    'mAP',
    'mAP_at_R',
    'precision_at_R',
    'recall_at_1',
    'timestamp',
    'folder_path',
]
columns = [col for col in columns if col in summary_df.columns]

sort_cols = [col for col in ['dataset_name', 'no_iso', 'timestamp'] if col in summary_df.columns]
if not sort_cols:
    sort_cols = [columns[0]] if columns else []

display_df = summary_df.sort_values(sort_cols).reset_index(drop=True) if sort_cols else summary_df
display(display_df[columns])


## 9. Demo: Compare CLIP vs IsoCLIP Top-K Retrieval

This final demo displays top-k image retrieval results from standard CLIP and IsoCLIP for both Caltech101 and CUB-200-2011. It reuses cached features from the previous retrieval runs; if the caches are missing, the feature extraction utilities will create them.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from PIL import Image

import utils as isoclip_utils
from data_utils import get_dataset
from encode_no_projection import get_projection_layers
from retrieval import apply_iso

DEMO_TOP_K = 5
DEMO_QUERIES_PER_DATASET = 3


def demo_safe_torch_device():
    if 'safe_torch_device' in globals():
        return safe_torch_device()
    if FORCE_CPU_RETRIEVAL:
        return torch.device('cpu')
    try:
        if torch.cuda.is_available():
            capability = torch.cuda.get_device_capability(0)
            arch = f'sm_{capability[0]}{capability[1]}'
            if arch in torch.cuda.get_arch_list():
                return torch.device('cuda')
            print(f'Using CPU in this demo because GPU arch {arch} is unsupported by the current PyTorch build.')
    except Exception as exc:
        print('CUDA compatibility check failed, using CPU:', repr(exc))
    return torch.device('cpu')


demo_device = demo_safe_torch_device()
isoclip_utils.device = demo_device
print('Device for demo:', demo_device)

clip_model, demo_clip_model_name, _ = isoclip_utils.load_clip(CLIP_MODEL_NAME, None, False, demo_device)
clip_model.eval()

W_image, W_text = get_projection_layers(clip_model, demo_clip_model_name)
W_image_for_iso = W_image.T
W_text_for_iso = W_text.T
_, W_image_iso = apply_iso(W_text_for_iso, W_image_for_iso, ISO_KTOP, ISO_KBOTTOM)

CUB_CLASS_LOOKUP = {}
classes_file = CUB_DIR / 'classes.txt'
if classes_file.exists():
    classes_df = pd.read_csv(classes_file, sep=' ', names=['target', 'class_name'])
    CUB_CLASS_LOOKUP = dict(zip(classes_df.target.astype(int), classes_df.class_name))


def dataset_record(dataset_name, dataset, index):
    if dataset_name == 'cub2011':
        row = dataset.data.iloc[index]
        label = int(row.target)
        return {
            'label': label,
            'class_name': CUB_CLASS_LOOKUP.get(label, str(label)),
            'path': CUB_DIR / 'images' / row.filepath,
            'filepath': row.filepath,
            'image_name': row.filepath.split('/')[1].split('.')[0],
        }
    if dataset_name == 'caltech101':
        item = dataset.data[index]
        path = Path(item.impath)
        return {
            'label': int(item.label),
            'class_name': str(item.classname).replace('_', ' '),
            'path': path,
            'filepath': str(path),
            'image_name': f'{path.parent.name}__{path.name}',
        }
    raise ValueError(f'Unsupported demo dataset: {dataset_name}')


def maybe_add_timm_bias_column(features):
    try:
        import open_clip
        timm_model_cls = getattr(getattr(open_clip, 'timm_model', None), 'TimmModel', None)
        if timm_model_cls is not None and isinstance(clip_model.visual, timm_model_cls):
            if demo_clip_model_name == 'PE-Core-B-16-meta' or 'EVA' in demo_clip_model_name:
                return features
            return torch.cat([features, features.new_ones(features.size(0), 1)], dim=1)
    except Exception as exc:
        print('Could not check OpenCLIP TimmModel bias handling:', repr(exc))
    return features


def project_iso_image_features(features):
    features = maybe_add_timm_bias_column(features.float().to(demo_device))
    return features @ W_image_iso.float().to(demo_device)


def get_demo_features(dataset_config, use_iso=False):
    dataset_name = dataset_config['dataset_name']
    query_split = dataset_config['query_split']
    gallery_split = dataset_config['gallery_split']
    query_features, query_names = isoclip_utils.get_features(
        DATA_ROOT, dataset_name, query_split, demo_clip_model_name, 'image',
        use_open_clip=False, open_clip_pretrained=None, use_iso=use_iso,
    )
    if gallery_split == query_split:
        gallery_features, gallery_names = query_features, query_names
    else:
        gallery_features, gallery_names = isoclip_utils.get_features(
            DATA_ROOT, dataset_name, gallery_split, demo_clip_model_name, 'image',
            use_open_clip=False, open_clip_pretrained=None, use_iso=use_iso,
        )
    return query_features, query_names, gallery_features, gallery_names


def topk_retrieval(query_features, gallery_features, query_index, top_k, gallery_names, excluded_gallery_name=None):
    scores = query_features[query_index] @ gallery_features.T
    scores = scores.detach().clone()
    if excluded_gallery_name is not None:
        for gallery_index, gallery_name in enumerate(gallery_names):
            if gallery_name == excluded_gallery_name:
                scores[gallery_index] = -torch.inf
    effective_top_k = min(top_k, int(torch.isfinite(scores).sum().item()))
    if effective_top_k <= 0:
        return [], []
    top_scores, top_indices = torch.topk(scores, k=effective_top_k)
    return top_indices.cpu().tolist(), top_scores.cpu().tolist()


def demo_ap_at_k(indices, gallery_labels, query_label, top_k, relevant_count):
    if relevant_count <= 0:
        return float('nan')
    hits = [gallery_labels[index] == query_label for index in indices[:top_k]]
    if not any(hits):
        return 0.0
    num_hits = 0
    precisions = []
    for rank, hit in enumerate(hits, start=1):
        if hit:
            num_hits += 1
            precisions.append(num_hits / rank)
    return sum(precisions) / min(top_k, relevant_count)


def first_correct_rank(indices, gallery_labels, query_label):
    for rank, index in enumerate(indices, start=1):
        if gallery_labels[index] == query_label:
            return rank
    return None


def demo_query_indices(dataset):
    if len(dataset) <= 0:
        return []
    raw_indices = [0, len(dataset) // 2, len(dataset) - 1]
    return list(dict.fromkeys(raw_indices))[:DEMO_QUERIES_PER_DATASET]


all_summary_rows = []

for dataset_config in DATASET_CONFIGS:
    dataset_name = dataset_config['dataset_name']
    display_name = dataset_config['display_name']
    query_split = dataset_config['query_split']
    gallery_split = dataset_config['gallery_split']

    display(Markdown(f'## {display_name} demo'))
    query_dataset = get_dataset(DATA_ROOT, dataset_name, query_split, preprocess=lambda image: image)
    gallery_dataset = get_dataset(DATA_ROOT, dataset_name, gallery_split, preprocess=lambda image: image)
    query_indices = demo_query_indices(query_dataset)
    demo_top_k = min(DEMO_TOP_K, len(gallery_dataset))

    print('Query split:', query_split, '| Gallery split:', gallery_split)
    print('Demo query indices:', query_indices)

    query_clip, query_clip_names, gallery_clip, gallery_clip_names = get_demo_features(dataset_config, use_iso=False)
    query_noproj, query_noproj_names, gallery_noproj, gallery_noproj_names = get_demo_features(dataset_config, use_iso=True)

    if query_clip_names != query_noproj_names:
        print(f'Warning: CLIP and IsoCLIP query feature caches have different ordering for {dataset_name}.')
    if gallery_clip_names != gallery_noproj_names:
        print(f'Warning: CLIP and IsoCLIP gallery feature caches have different ordering for {dataset_name}.')

    for query_index in query_indices:
        expected_name = dataset_record(dataset_name, query_dataset, query_index)['image_name']
        if query_clip_names[query_index] != expected_name:
            print(f'Warning: query feature order may not match {dataset_name} at query index {query_index}.')

    clip_query_features = F.normalize(query_clip.float().to(demo_device), dim=1)
    clip_gallery_features = F.normalize(gallery_clip.float().to(demo_device), dim=1)
    iso_query_features = F.normalize(project_iso_image_features(query_noproj), dim=1)
    iso_gallery_features = F.normalize(project_iso_image_features(gallery_noproj), dim=1)

    gallery_labels = [dataset_record(dataset_name, gallery_dataset, i)['label'] for i in range(len(gallery_dataset))]
    same_split = query_split == gallery_split
    retrieval_cache = {}
    summary_rows = []

    for query_index in query_indices:
        query_info = dataset_record(dataset_name, query_dataset, query_index)
        excluded_name = query_info['image_name'] if same_split else None
        relevant_count = sum(label == query_info['label'] for label in gallery_labels)
        if same_split:
            relevant_count -= sum(
                gallery_name == excluded_name and gallery_labels[i] == query_info['label']
                for i, gallery_name in enumerate(gallery_clip_names)
            )

        clip_indices, clip_scores = topk_retrieval(
            clip_query_features, clip_gallery_features, query_index, demo_top_k,
            gallery_clip_names, excluded_gallery_name=excluded_name,
        )
        iso_indices, iso_scores = topk_retrieval(
            iso_query_features, iso_gallery_features, query_index, demo_top_k,
            gallery_noproj_names, excluded_gallery_name=excluded_name,
        )
        retrieval_cache[query_index] = {'CLIP': (clip_indices, clip_scores), 'IsoCLIP': (iso_indices, iso_scores)}

        for method, indices, _scores in [('CLIP', clip_indices, clip_scores), ('IsoCLIP', iso_indices, iso_scores)]:
            correct_at_k = sum(gallery_labels[index] == query_info['label'] for index in indices)
            row = {
                'dataset': dataset_name,
                'query_split': query_split,
                'gallery_split': gallery_split,
                'query_index': query_index,
                'query_class': query_info['label'],
                'query_class_name': query_info['class_name'],
                'method': method,
                f'correct@{demo_top_k}': correct_at_k,
                'first_correct_rank': first_correct_rank(indices, gallery_labels, query_info['label']),
                f'demo_AP@{demo_top_k}': round(demo_ap_at_k(indices, gallery_labels, query_info['label'], demo_top_k, relevant_count), 4),
                'query_filepath': query_info['filepath'],
            }
            summary_rows.append(row)
            all_summary_rows.append(row)

    summary_demo_df = pd.DataFrame(summary_rows)
    display(Markdown('**Summary across demo queries**'))
    display(summary_demo_df)

    for example_no, query_index in enumerate(query_indices, start=1):
        query_info = dataset_record(dataset_name, query_dataset, query_index)
        display(Markdown(f'### Demo example {example_no}: query index {query_index}'))
        print('Query path:', query_info['path'])
        print('Query class:', query_info['label'], query_info['class_name'])
        display(Image.open(query_info['path']).convert('RGB').resize((256, 256)))

        rows = []
        for method, (indices, scores) in retrieval_cache[query_index].items():
            for rank, (index, score) in enumerate(zip(indices, scores), start=1):
                gallery_info = dataset_record(dataset_name, gallery_dataset, index)
                rows.append({
                    'method': method,
                    'rank': rank,
                    'score': round(float(score), 4),
                    'same_class': gallery_info['label'] == query_info['label'],
                    'target': gallery_info['label'],
                    'class_name': gallery_info['class_name'],
                    'filepath': gallery_info['filepath'],
                })

        display(pd.DataFrame(rows))

        fig, axes = plt.subplots(2, demo_top_k, figsize=(3.0 * demo_top_k, 6.0), squeeze=False)
        for row_id, (method, (indices, scores)) in enumerate(retrieval_cache[query_index].items()):
            for col_id, (index, score) in enumerate(zip(indices, scores)):
                ax = axes[row_id][col_id]
                gallery_info = dataset_record(dataset_name, gallery_dataset, index)
                image = Image.open(gallery_info['path']).convert('RGB')
                same_class = gallery_info['label'] == query_info['label']
                ax.imshow(image)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_title(f"#{col_id + 1}  score={score:.3f}\nclass={gallery_info['label']}", fontsize=9)
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_linewidth(3 if same_class else 1)
                    spine.set_edgecolor('#2e7d32' if same_class else '#b0b0b0')
            axes[row_id][0].set_ylabel(method, fontsize=12, rotation=90, labelpad=18)

        fig.suptitle(
            f"{display_name}: Top-{demo_top_k} retrieval for query class {query_info['label']}: {query_info['class_name']}",
            y=1.02,
        )
        plt.tight_layout()
        plt.show()

if all_summary_rows:
    display(Markdown('## Combined demo summary'))
    display(pd.DataFrame(all_summary_rows))
